# Módulo 03 · Aula 1 — Matplotlib

**Capacitação Introdutória de Ciência de Dados · FEA.dev**

---

Uma tabela com 10 mil linhas não cabe na cabeça de ninguém. Um gráfico, sim.

Gráficos servem para duas coisas bem diferentes, e vale distinguir desde já:

- **explorar** — gráficos rápidos e feios, feitos para *você* entender os dados;
- **comunicar** — gráficos caprichados, feitos para *outra pessoa* entender sua conclusão.

Nesta aula você aprende o **Matplotlib**, a biblioteca base de visualização em Python.
Ela é mais verbosa que as alternativas, mas é a fundação: tudo o mais (inclusive o
Seaborn, da próxima aula) é construído em cima dela, e é a ela que você recorre quando
precisa de controle fino.

Ao final você vai saber:

- a estrutura **figura → eixos** e por que ela importa;
- fazer gráficos de linha, barra, histograma e dispersão;
- rotular tudo (título, eixos, legenda) — o que separa um gráfico de um rabisco;
- montar painéis com vários gráficos e salvar o resultado em arquivo.

**Tempo estimado:** 60 minutos.

### Antes de começar — se você está no Google Colab

Este notebook lê arquivos da pasta `data/` do repositório, e no Colab a máquina começa vazia. **Execute a célula abaixo antes de qualquer outra**: ela traz o repositório e entra na pasta deste módulo, de modo que os caminhos `../data/...` usados no material funcionem sem alteração.

No VS Code ou no Jupyter local a célula não faz nada — os arquivos já estão no seu disco.

In [ ]:
# Setup do Google Colab.
# Traz o repositório da capacitação e entra na pasta deste módulo, para que os
# caminhos "../data/..." usados no material funcionem sem nenhuma alteração.
# Fora do Colab (VS Code, Jupyter local) esta célula não faz nada.
# Pode ser executada mais de uma vez sem problema.
import os
import subprocess
import sys

PASTA_DESTE_MODULO = "03_Visualizacao_EDA"
REPOSITORIO = "https://github.com/gustavokatsuo/Introducao-a-Ciencia-de-Dados.git"

if "google.colab" in sys.modules and not os.path.isdir("../data"):
    destino = "/content/Introducao-a-Ciencia-de-Dados"
    if not os.path.isdir(destino):
        print("Baixando o material da capacitação...")
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORIO, destino], check=True)
    os.chdir(os.path.join(destino, PASTA_DESTE_MODULO))
    print("Pronto. Pasta de trabalho:", os.getcwd())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline

acoes = pd.read_csv("../data/acoes_b3.csv", parse_dates=["data"])
ibovespa = pd.read_csv("../data/ibovespa.csv", parse_dates=["data"])
indicadores = pd.read_csv("../data/indicadores_macro.csv", parse_dates=["data"])
empresas = pd.read_csv("../data/empresas_b3.csv")

print("Dados carregados:", acoes.shape, ibovespa.shape, indicadores.shape)

## 1. Figura e eixos

O Matplotlib tem duas peças:

- a **figura** (`fig`) é a folha de papel — o retângulo inteiro, com seu tamanho;
- os **eixos** (`ax`) são a área de desenho dentro dela, com seu par de escalas.

Uma figura pode conter vários eixos (é assim que se fazem painéis). Quase tudo que você
vai querer configurar — título, rótulos, limites, grade — é um método do `ax`.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))     # uma figura com um par de eixos

ax.plot([1, 2, 3, 4], [10, 14, 12, 18])

plt.show()

> Você vai encontrar por aí um estilo alternativo, `plt.plot(...)` direto, sem criar
> `fig` e `ax`. Funciona para gráficos rápidos, mas se confunde quando há mais de um
> gráfico. **Use sempre `fig, ax = plt.subplots()`** — é o estilo recomendado pela
> própria documentação e o único que escala.

## 2. Gráfico de linha: evolução no tempo

O uso natural da linha é mostrar como algo **evolui**. Ela sugere continuidade entre os
pontos — por isso não a use para categorias.

In [ ]:
petr = acoes[acoes["ticker"] == "PETR4"].sort_values("data")

fig, ax = plt.subplots(figsize=(11, 4.5))

ax.plot(petr["data"], petr["fechamento_ajustado"])

plt.show()

Esse gráfico já mostra alguma coisa, mas ele não se explica. Quem é essa linha? Em que
unidade? De que período? **Um gráfico sem rótulos não comunica nada** — e daqui a duas
semanas nem você vai lembrar.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))

ax.plot(petr["data"], petr["fechamento_ajustado"], color="#1f4e79", linewidth=1.4)

ax.set_title("PETR4 — preço de fechamento ajustado (2021–2025)", fontsize=13, pad=12)
ax.set_xlabel("Data")
ax.set_ylabel("Preço ajustado (R$)")
ax.grid(alpha=0.3)

plt.show()

### Várias linhas e legenda

Para comparar séries, plote uma por vez no mesmo `ax` e dê um `label` a cada uma.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

for ticker in ["PETR4", "VALE3", "ITUB4", "WEGE3"]:
    serie = acoes[acoes["ticker"] == ticker].sort_values("data")
    ax.plot(serie["data"], serie["fechamento_ajustado"], label=ticker, linewidth=1.2)

ax.set_title("Preço ajustado de quatro ações da B3 (2021–2025)", fontsize=13, pad=12)
ax.set_xlabel("Data")
ax.set_ylabel("Preço ajustado (R$)")
ax.legend()
ax.grid(alpha=0.3)

plt.show()

Esse gráfico tem um problema **conceitual**, não estético: os quatro ativos têm níveis
de preço diferentes, então o eixo vertical mede coisas que não são comparáveis. VALE3
parece "melhor" só por custar mais caro.

A correção é comparar **retorno acumulado**: quanto valeria hoje R$ 1 investido no
primeiro dia. Aí todas as séries começam em 100 e a comparação passa a ser justa.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

for ticker in ["PETR4", "VALE3", "ITUB4", "WEGE3"]:
    serie = acoes[acoes["ticker"] == ticker].sort_values("data")
    base = serie["fechamento_ajustado"].iloc[0]
    ax.plot(serie["data"], serie["fechamento_ajustado"] / base * 100,
            label=ticker, linewidth=1.3)

ax.axhline(100, color="gray", linestyle="--", linewidth=1)   # linha de referência
ax.set_title("Retorno acumulado — base 100 no primeiro pregão de 2021",
             fontsize=13, pad=12)
ax.set_xlabel("Data")
ax.set_ylabel("Índice (100 = início)")
ax.legend()
ax.grid(alpha=0.3)

plt.show()

> **A escolha do que plotar é uma decisão analítica.** Os dois gráficos acima usam os
> mesmos dados e a mesma técnica, mas só o segundo responde à pergunta "qual foi o melhor
> investimento?". Antes de escolher o tipo de gráfico, decida qual é a pergunta.

## 3. Gráfico de barras: comparar categorias

Barras comparam **quantidades entre categorias**. O olho humano compara comprimentos com
muita precisão, o que faz das barras o gráfico mais legível que existe para essa tarefa.

In [ ]:
retorno_total = []
for ticker in sorted(acoes["ticker"].unique()):
    serie = acoes[acoes["ticker"] == ticker].sort_values("data")["fechamento_ajustado"]
    retorno_total.append({
        "ticker": ticker,
        "retorno": (serie.iloc[-1] / serie.iloc[0] - 1) * 100,
    })

retorno_total = pd.DataFrame(retorno_total).sort_values("retorno")
retorno_total

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

cores = ["#c0392b" if valor < 0 else "#27ae60" for valor in retorno_total["retorno"]]
ax.barh(retorno_total["ticker"], retorno_total["retorno"], color=cores)

ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Retorno acumulado no período 2021–2025", fontsize=13, pad=12)
ax.set_xlabel("Retorno (%)")
ax.grid(axis="x", alpha=0.3)

# Escrevendo o valor ao lado de cada barra
for posicao, valor in enumerate(retorno_total["retorno"]):
    deslocamento = 3 if valor >= 0 else -3
    alinhamento = "left" if valor >= 0 else "right"
    ax.text(valor + deslocamento, posicao, f"{valor:.0f}%",
            va="center", ha=alinhamento, fontsize=10)

plt.show()

Três decisões deliberadas nesse gráfico, e todas melhoram a leitura:

1. **Barras horizontais** (`barh`) — os rótulos ficam legíveis sem girar o texto;
2. **Ordenação por valor** — o olho encontra o maior e o menor imediatamente;
3. **Cor com significado** — vermelho para negativo, verde para positivo. Cor só deve
   ser usada quando codifica informação; cor decorativa atrapalha.

## 4. Histograma: como uma variável se distribui

O histograma divide os valores em faixas e conta quantas observações caem em cada uma.
É como se descobre o **formato** de uma variável: onde ela se concentra, se é simétrica,
se tem valores extremos.

In [ ]:
petr = petr.sort_values("data").copy()
petr["retorno_diario"] = petr["fechamento_ajustado"].pct_change() * 100

fig, ax = plt.subplots(figsize=(9, 4.5))

ax.hist(petr["retorno_diario"].dropna(), bins=50, color="#1f4e79",
        edgecolor="white", alpha=0.85)

ax.axvline(0, color="black", linewidth=1)
ax.set_title("Distribuição dos retornos diários da PETR4 (2021–2025)",
             fontsize=13, pad=12)
ax.set_xlabel("Retorno diário (%)")
ax.set_ylabel("Número de pregões")
ax.grid(axis="y", alpha=0.3)

plt.show()

> `pct_change()` calcula a variação percentual em relação à linha anterior — é o cálculo
> de retorno que fizemos "na mão" com fatiamento na aula de NumPy, agora em um método.
> A primeira linha vira `NaN`, porque não existe dia anterior; daí o `.dropna()`.

O número de faixas (`bins`) muda o que você enxerga. Poucas faixas escondem estrutura;
muitas transformam o gráfico em ruído. Vale sempre testar alguns valores:

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(14, 3.8))

for ax, n_bins in zip(eixos, [8, 30, 120]):
    ax.hist(petr["retorno_diario"].dropna(), bins=n_bins,
            color="#1f4e79", edgecolor="white")
    ax.set_title(f"bins = {n_bins}")
    ax.set_xlabel("Retorno diário (%)")

eixos[0].set_ylabel("Número de pregões")
fig.suptitle("O mesmo dado, três resoluções", fontsize=13)
fig.tight_layout()

plt.show()

## 5. Dispersão: relação entre duas variáveis

O gráfico de dispersão (*scatter*) coloca uma variável em cada eixo e um ponto por
observação. É a ferramenta para investigar **se duas coisas andam juntas**.

In [ ]:
# Retorno diário da PETR4 contra o do Ibovespa, no mesmo dia
ibov = ibovespa.sort_values("data").copy()
ibov["retorno_ibov"] = ibov["fechamento"].pct_change() * 100

comparacao = petr[["data", "retorno_diario"]].merge(
    ibov[["data", "retorno_ibov"]], on="data", how="inner"
).dropna()

fig, ax = plt.subplots(figsize=(6.5, 6))

ax.scatter(comparacao["retorno_ibov"], comparacao["retorno_diario"],
           alpha=0.35, s=16, color="#1f4e79")

ax.axhline(0, color="gray", linewidth=0.8)
ax.axvline(0, color="gray", linewidth=0.8)
ax.set_title("PETR4 × Ibovespa — retornos diários", fontsize=13, pad=12)
ax.set_xlabel("Retorno diário do Ibovespa (%)")
ax.set_ylabel("Retorno diário da PETR4 (%)")
ax.grid(alpha=0.3)

plt.show()

A nuvem inclinada para cima mostra o esperado: quando a bolsa sobe, a PETR4 tende a
subir. Note o uso de `alpha=0.35` — com mil pontos sobrepostos, a transparência revela
onde há concentração. Sem ela, o miolo vira uma mancha sólida.

## 6. Painéis: vários gráficos em uma figura

`plt.subplots(linhas, colunas)` devolve uma matriz de eixos. Painéis são a melhor forma
de comparar o mesmo recorte entre vários grupos.

In [ ]:
tickers = ["PETR4", "VALE3", "ITUB4", "WEGE3"]

fig, eixos = plt.subplots(2, 2, figsize=(13, 7))
eixos = eixos.flatten()          # transforma a matriz 2×2 em uma lista de 4

for ax, ticker in zip(eixos, tickers):
    serie = acoes[acoes["ticker"] == ticker].sort_values("data")
    ax.plot(serie["data"], serie["fechamento_ajustado"], color="#1f4e79", linewidth=1.1)
    ax.set_title(ticker, fontsize=12)
    ax.set_ylabel("R$")
    ax.grid(alpha=0.3)

fig.suptitle("Preço ajustado por ativo (2021–2025)", fontsize=14)
fig.tight_layout()

plt.show()

> **Atenção:** Repare que **cada painel tem sua própria escala vertical**. Isso é ótimo
> para ver o formato de cada série, e péssimo para comparar níveis entre elas. Se a
> comparação for o objetivo, use `plt.subplots(..., sharey=True)` — ou volte ao gráfico
> único com base 100. Escala é uma das principais formas de um gráfico enganar, inclusive
> sem querer.

## 7. Ajustes que valem a pena

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

serie = indicadores.sort_values("data")

ax.bar(serie["data"], serie["ipca_mes_pct"], width=20,
       color="#c55a11", label="IPCA no mês")
ax.plot(serie["data"], serie["selic_mes_pct"], color="#1f4e79",
        linewidth=2, label="Selic no mês")

ax.set_title("IPCA e Selic mensais (2021–2025)", fontsize=13, pad=12)
ax.set_xlabel("Data")
ax.set_ylabel("% ao mês")
ax.legend(loc="upper right", frameon=True)
ax.grid(axis="y", alpha=0.3)
ax.axhline(0, color="black", linewidth=0.8)

# Removendo as bordas superior e direita — menos tinta, mais dado
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.show()

Uma referência rápida dos ajustes mais usados:

| Comando | Efeito |
|---|---|
| `ax.set_title(...)`, `ax.set_xlabel(...)`, `ax.set_ylabel(...)` | rótulos |
| `ax.legend()` | legenda (exige `label=` nas séries) |
| `ax.grid(alpha=0.3)` | grade discreta |
| `ax.set_xlim(a, b)` / `ax.set_ylim(a, b)` | limites dos eixos |
| `ax.axhline(y)` / `ax.axvline(x)` | linha de referência |
| `ax.tick_params(axis="x", rotation=45)` | girar rótulos do eixo |
| `fig.tight_layout()` | ajustar espaçamentos automaticamente |
| `fig.savefig("nome.png", dpi=150, bbox_inches="tight")` | salvar em arquivo |

In [ ]:
# Salvando uma figura
from pathlib import Path
import tempfile

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(petr["data"], petr["fechamento_ajustado"], color="#1f4e79")
ax.set_title("PETR4 — fechamento ajustado")
ax.set_xlabel("Data")
ax.set_ylabel("R$")

destino = Path(tempfile.gettempdir()) / "petr4.png"
fig.savefig(destino, dpi=150, bbox_inches="tight")
plt.close(fig)          # fecha a figura para ela não ser exibida duas vezes

print(f"Figura salva ({destino.stat().st_size/1024:.0f} KB)")

> `bbox_inches="tight"` corta o espaço branco em volta; `dpi=150` dá resolução decente
> para colar em um relatório. Em uma apresentação, use `dpi=200` ou mais.

## 8. Sete erros que estragam um gráfico

1. **Sem título e sem rótulos nos eixos.** O erro mais comum e o mais fácil de evitar.
2. **Unidade ausente.** "Retorno" pode ser 0,05 ou 5%. Escreva no rótulo.
3. **Eixo Y que não começa em zero, em gráfico de barras.** Isso amplia diferenças
   pequenas artificialmente. Em gráficos de linha, cortar o eixo é aceitável e comum.
4. **Categorias sem ordenação.** Ordene por valor, salvo quando houver ordem natural
   (meses, faixas etárias).
5. **Excesso de cor.** Cor deve codificar informação. Se todas as barras são de
   categorias iguais, use uma cor só.
6. **Tipo errado de gráfico.** Linha para categorias e pizza com dez fatias são os
   campeões.
7. **Escalas diferentes em painéis comparativos.** Compare maçãs com maçãs.

> Um teste honesto: **mostre o gráfico para alguém que não viu seus dados.** Se essa
> pessoa não conseguir dizer o que ele mostra em dez segundos, o gráfico ainda não está
> pronto.

## 9. Recapitulando

- Estrutura: `fig, ax = plt.subplots(figsize=(largura, altura))`. Configure tudo pelo
  `ax`.
- `ax.plot` (evolução), `ax.bar`/`ax.barh` (categorias), `ax.hist` (distribuição),
  `ax.scatter` (relação entre duas variáveis).
- Título, rótulos com unidade e legenda **sempre**.
- `plt.subplots(linhas, colunas)` monta painéis; cuidado com escalas diferentes.
- `fig.savefig(..., dpi=150, bbox_inches="tight")` exporta.
- A escolha do gráfico e da transformação (nível × base 100) é decisão analítica, não
  estética.

**Próxima aula:** Seaborn — os mesmos gráficos com muito menos código, e alguns que o
Matplotlib puro não oferece.